In [ ]:
from sklearn.model_selection import train_test_split

all_paths = file_list
all_labels = labels

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths,
    all_labels,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_dataset = AudioDataset(train_paths, train_labels)
val_dataset = AudioDataset(val_paths, val_labels)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = FullModel(64, 100).to(device)

import os
if os.path.exists("best.pth"):
    model.load_state_dict(torch.load("best.pth", map_location=device))

sinc = SincConv().to(device)
encoder = Encoder().to(device)

optimizer = torch.optim.Adam(
    list(model.parameters()) +
    list(sinc.parameters()) +
    list(encoder.parameters()),
    lr=0.0003
)

criterion = nn.CrossEntropyLoss()

In [ ]:
def validate(model, val_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    correct = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            x = sinc(x)
            x = encoder(x)
            x = F.adaptive_max_pool1d(x, 100)

            outputs = model(x)
            loss = criterion(outputs, y)

            batch_size = x.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            #  accuracy
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()

    avg_loss = total_loss / total_samples
    val_acc = correct / total_samples

    return avg_loss, val_acc

In [ ]:
best_loss = float('inf')
best_val_acc = 0
for epoch in range(45):

    # -------- TRAIN --------
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        x = sinc(x)
        x = encoder(x)
        x = F.adaptive_max_pool1d(x, 100)

        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # accuracy
        _, predicted = torch.max(out, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total


    # -------- VALIDATION --------
    val_loss, val_acc = validate(model, val_loader, criterion, device)


    # -------- SAVE BEST --------

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best model")


    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")